# Building a Blockchain from Scratch

## Welcome!

In this workshop, we are **not** going to use an existing blockchain like Bitcoin or Ethereum.

Instead, we're going to build our own blockchain from scratch using Python.

By the end of this workshop, you'll understand:

- What a blockchain actually is
- Why blockchains exist
- How transactions are stored
- How blocks are connected together
- Why hashes make data tamper-resistant
- How mining works
- How attackers try to break blockchains

Don't worry if you've never worked with blockchain before.

We'll build every component ourselves.

In [ ]:
# A traditional banking system might store balances like this.

bank_database = {
    "Alice": 100,
    "Bob": 50,
    "Charlie": 25
}

print("Before transaction:")
print(bank_database)

# Alice sends Bob $20

bank_database["Alice"] -= 20
bank_database["Bob"] += 20

print("\nAfter transaction:")
print(bank_database)

Before transaction:
{'Alice': 100, 'Bob': 50, 'Charlie': 25}

After transaction:
{'Alice': 80, 'Bob': 70, 'Charlie': 25}


What happens if someone edits this dictionary directly?

In [ ]:
bank_database["Alice"] = 1000000

print(bank_database)

{'Alice': 1000000, 'Bob': 70, 'Charlie': 25}


# 📦 Section 2 – Building Our First Block

Before we build a blockchain, we first need to build a **block**.

A block is simply a container that stores information. Every block in our blockchain will contain:

- **Index** – The block's position in the blockchain.
- **Transactions** – A list of transactions stored in the block.
- **Previous Hash** – A reference to the previous block (we'll generate real hashes later).
- **Timestamp** – The time when the block was created.

## 🌱 Genesis Block

Every blockchain begins with a special first block called the **Genesis Block**.

Since there is no block before it, its `previous_hash` is set to `"0"`.

## 🎯 What We'll Learn

By the end of this section, you'll be able to:
- Create a block using a Python class.
- Create the Genesis Block.
- Create additional blocks with transactions.
- Understand why blocks alone are **not enough** to secure data.

In [ ]:
import time

class Block:
    """
    Represents a single block in the blockchain.
    """

    def __init__(self, index, transactions, previous_hash):
        self.index = index                      # Position of the block
        self.transactions = transactions        # Transactions stored in this block
        self.previous_hash = previous_hash      # Link to previous block
        self.timestamp = time.time()            # Creation time

    def __str__(self):
        return f"""
==========================
Block #{self.index}
==========================
Transactions : {self.transactions}
Previous Hash: {self.previous_hash}
Timestamp    : {self.timestamp}
"""


# -----------------------------
# Create the Genesis Block
# -----------------------------
genesis_block = Block(
    index=0,
    transactions=[],
    previous_hash="0"
)

print("Genesis Block")
print(genesis_block)


# -----------------------------
# Create another block
# -----------------------------
block_1 = Block(
    index=1,
    transactions=[
        "Alice -> Bob : $20",
        "Bob -> Charlie : $5"
    ],
    previous_hash="Placeholder"
)

print(block_1)


# -----------------------------
# Demonstrate the Problem
# -----------------------------
print("\n--- Modifying a Transaction ---")

block_1.transactions[0] = "Alice -> Bob : $2000"

print(block_1)

Genesis Block

Block #0
Transactions : []
Previous Hash: 0
Timestamp    : 1785787960.142366


Block #1
Transactions : ['Alice -> Bob : $20', 'Bob -> Charlie : $5']
Previous Hash: Placeholder
Timestamp    : 1785787960.1425948


--- Modifying a Transaction ---

Block #1
Transactions : ['Alice -> Bob : $2000', 'Bob -> Charlie : $5']
Previous Hash: Placeholder
Timestamp    : 1785787960.1425948



Notice that Python allows us to change the block after it has been created.
### How can we detect this kind of tampering?
We'll solve this in the below section using cryptographic hashing.

# 🔐 Section 3 – Cryptographic Hashing

So far, our blocks can be modified after they are created.

How can we detect if someone changes the contents of a block?

The solution is a **cryptographic hash**.

A hash is a unique digital fingerprint generated from data.

### Properties of a Cryptographic Hash

- The same input always produces the same hash.
- Even a tiny change in the input creates a completely different hash.
- It is computationally infeasible to reverse a hash to recover the original data.

In this section, we'll generate a hash for each block so we can detect if its contents have been modified.

In [ ]:
import hashlib

message = "Hello Blockchain"

hash_value = hashlib.sha256(message.encode()).hexdigest()

print("Message:", message)
print("SHA-256 Hash:")
print(hash_value)

Message: Hello Blockchain
SHA-256 Hash:
7cf88f2ee398c0b7c0e760a1dccaf3571e0baccf310f11fe3bdfd0b09675ea75


Below, we can see "Hello" and "hello" and see the different hashes generated

In [ ]:
import hashlib

message1 = "Hello Blockchain"
message2 = "hello Blockchain"

hash1 = hashlib.sha256(message1.encode()).hexdigest()
hash2 = hashlib.sha256(message2.encode()).hexdigest()

print("Message 1:", message1)
print(hash1)

print()

print("Message 2:", message2)
print(hash2)

Message 1: Hello Blockchain
7cf88f2ee398c0b7c0e760a1dccaf3571e0baccf310f11fe3bdfd0b09675ea75

Message 2: hello Blockchain
7e4c73bd719b21390ac6cdc3c4c5cea8fab62141b512906c152b02da210452b9


## Now, let's hash the block

In [ ]:
import hashlib
import json
import time


class Block:

    def __init__(self, index, transactions, previous_hash):
        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.timestamp = time.time()

        self.hash = self.calculate_hash()

    def calculate_hash(self):

        block_data = {
            "index": self.index,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "timestamp": self.timestamp
        }

        encoded_block = json.dumps(
            block_data,
            sort_keys=True
        ).encode()

        return hashlib.sha256(encoded_block).hexdigest()

    def __str__(self):

        return f"""
=========================
Block #{self.index}
=========================

Transactions : {self.transactions}

Previous Hash:
{self.previous_hash}

Current Hash:
{self.hash}

Timestamp:
{self.timestamp}
"""

Create a Block

In [ ]:
genesis = Block(
    0,
    [],
    "0"
)

print(genesis)


Block #0

Transactions : []

Previous Hash:
0

Current Hash:
882077f6e94d8a8fecd34ad2ce6c25dc18bbc7b1eddd2089ce9c6d6aac19d7c3

Timestamp:
1785787960.1767893



An attacker can still modify the data and calculate a new Hash. As, we haven't updated the stored hash.

In [ ]:
print("Original Hash")
print(genesis.hash)

print()

genesis.transactions.append("Alice -> Bob : $50")

print("Stored Hash")
print(genesis.hash)

print()

print("Recalculated Hash")
print(genesis.calculate_hash())

Original Hash
882077f6e94d8a8fecd34ad2ce6c25dc18bbc7b1eddd2089ce9c6d6aac19d7c3

Stored Hash
882077f6e94d8a8fecd34ad2ce6c25dc18bbc7b1eddd2089ce9c6d6aac19d7c3

Recalculated Hash
2547292e1473bddca7ac61249e6a738be563ca6c550aaa11d3d9aae4f05c8f89


Let's automate this using a function. We cam verify if a block has been tampered or not.

In [ ]:
def verify_block(block):
    return block.hash == block.calculate_hash()


print("Before Modification")
print(verify_block(genesis))

genesis.transactions.append("Charlie -> David : $25")

print()

print("After Modification")
print(verify_block(genesis))

Before Modification
False

After Modification
False


What if an attacker modifies the block and then simply recalculates and replaces the stored hash?

#  Section 4 – Linking Blocks with Previous Hash

So far, each block can verify its own integrity using a cryptographic hash.

However, an attacker could still modify the block and generate a new valid hash.

To make tampering much harder, every block stores the **hash of the previous block**.

This creates a chain of connected blocks:

Genesis Block → Block 1 → Block 2 → Block 3

If one block changes, its hash changes.

Since the next block stores that hash, it immediately becomes invalid.

This is what makes a **blockchain**.

In [ ]:
# Create the Genesis Block
genesis = Block(
    index=0,
    transactions=["Genesis Block"],
    previous_hash="0"
)

# Create Block 1
block1 = Block(
    index=1,
    transactions=[
        "Alice -> Bob : $20"
    ],
    previous_hash=genesis.hash
)

# Create Block 2
block2 = Block(
    index=2,
    transactions=[
        "Bob -> Charlie : $10"
    ],
    previous_hash=block1.hash
)

print(genesis)
print(block1)
print(block2)


Block #0

Transactions : ['Genesis Block']

Previous Hash:
0

Current Hash:
e1777d1b4f939fdbad24adb5a29aab70df2f363910ff6ab00eae481530f48c43

Timestamp:
1785787960.2043533


Block #1

Transactions : ['Alice -> Bob : $20']

Previous Hash:
e1777d1b4f939fdbad24adb5a29aab70df2f363910ff6ab00eae481530f48c43

Current Hash:
35221173bdd0d2ef177b055345ec19b7e2e3cd4799fecdd223c07c270e92660e

Timestamp:
1785787960.2044678


Block #2

Transactions : ['Bob -> Charlie : $10']

Previous Hash:
35221173bdd0d2ef177b055345ec19b7e2e3cd4799fecdd223c07c270e92660e

Current Hash:
7564341c9d72746c839c89f35dad8a57a426c3a045e09485f7d5d1ad7d0d1e7e

Timestamp:
1785787960.2045395



Let's store them in a List

In [ ]:
blockchain = [
    genesis,
    block1,
    block2
]

print("Blockchain Length:", len(blockchain))

Blockchain Length: 3


# Let's now verify the Entire Chain

In [ ]:
def verify_chain(chain):

    for i in range(1, len(chain)):

        current = chain[i]
        previous = chain[i - 1]

        # Verify current block wasn't modified
        if current.hash != current.calculate_hash():
            return False

        # Verify it points to the correct previous block
        if current.previous_hash != previous.hash:
            return False

    return True

In [ ]:
print(verify_chain(blockchain))

True


#  Section 5 – Transactions

So far, our blocks have stored simple strings like:

```python
"Alice -> Bob : $20"
```

While this works, it's not how real blockchains operate.

Instead, blockchains store **transactions** structured records that describe the movement of value between users.

Each transaction contains:

- **Transaction ID** – A unique identifier.
- **Sender** – The person sending funds.
- **Receiver** – The person receiving funds.
- **Amount** – The value being transferred.
- **Timestamp** – When the transaction was created.

A block simply stores a **list of transactions**.

> **Note:** We are **not** verifying ownership yet. At the moment, anyone could create a transaction claiming to be Alice. We'll solve this in the next section using **digital signatures**.

In [ ]:
import json
import hashlib
import uuid
import time

class Block:

    def __init__(self, index, transactions, previous_hash):
        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.timestamp = time.time()

        self.hash = self.calculate_hash()

    def calculate_hash(self):

      block_data = {
          "index": self.index,
          "transactions": [tx.to_dict() for tx in self.transactions],
          "previous_hash": self.previous_hash,
          "timestamp": self.timestamp
      }

      encoded_block = json.dumps(
          block_data,
          sort_keys=True
      ).encode()

      return hashlib.sha256(encoded_block).hexdigest()
class Transaction:
    def __init__(self, sender, receiver, amount):
        self.transaction_id = str(uuid.uuid4())
        self.sender = sender
        self.receiver = receiver
        self.amount = amount
        self.timestamp = time.time()

    def to_dict(self):
        return {
            "transaction_id": self.transaction_id,
            "sender": self.sender,
            "receiver": self.receiver,
            "amount": self.amount,
            "timestamp": self.timestamp
        }

    def __str__(self):
        return f"""
Transaction ID : {self.transaction_id}
Sender         : {self.sender}
Receiver       : {self.receiver}
Amount         : ${self.amount}
Timestamp      : {self.timestamp}
"""


# -----------------------------------------
# Create Sample Transactions
# -----------------------------------------

tx1 = Transaction("Alice", "Bob", 20)
tx2 = Transaction("Bob", "Charlie", 10)
tx3 = Transaction("Charlie", "David", 5)

print("Transaction 1")
print(tx1)

print("Transaction 2")
print(tx2)

print("Transaction 3")
print(tx3)


# -----------------------------------------
# Add Transactions to a Block
# -----------------------------------------

block = Block(
    index=3,
    transactions=[tx1, tx2, tx3],
    previous_hash="Previous_Block_Hash"
)

print("\n==========================")
print("Block Containing Transactions")
print("==========================")
print(block)


# -----------------------------------------
# Security Demonstration
# -----------------------------------------

print("\nCan anyone create a fake transaction?")

fake_tx = Transaction(
    "Alice",
    "Mallory",
    1000000
)

print(fake_tx)

print("\n⚠️ Problem:")
print("Our blockchain accepts this transaction because")
print("we haven't verified that Alice actually created it.")

print("\n➡️ In the next section, we'll introduce Digital Signatures")
print("to prove transaction ownership.")

Transaction 1

Transaction ID : b935ba89-6ea4-4032-9dae-7e52da6179d5
Sender         : Alice
Receiver       : Bob
Amount         : $20
Timestamp      : 1785787960.2421608

Transaction 2

Transaction ID : a9674ba3-b7cf-46b8-9652-11a0cb46431b
Sender         : Bob
Receiver       : Charlie
Amount         : $10
Timestamp      : 1785787960.2422602

Transaction 3

Transaction ID : 08c07211-4e55-46b1-b30a-3a7ec25c622a
Sender         : Charlie
Receiver       : David
Amount         : $5
Timestamp      : 1785787960.2426584


Block Containing Transactions

Can anyone create a fake transaction?

Transaction ID : 704dd81a-2e3e-48a3-9eb8-1fd86f1b41f3
Sender         : Alice
Receiver       : Mallory
Amount         : $1000000
Timestamp      : 1785787960.24467


⚠️ Problem:
Our blockchain accepts this transaction because
we haven't verified that Alice actually created it.

➡️ In the next section, we'll introduce Digital Signatures
to prove transaction ownership.


How do we really prove this transactions were created ?

# ✍️ Section 6 – Wallets & Digital Signatures

In the previous section, anyone could create a transaction like:

```text
Alice → Mallory : 1,000,000 coins
```

Our blockchain had no way of knowing whether Alice actually created it.

This is solved using **public-key cryptography**.

Every user owns:

- **Private Key** – A secret key used to sign transactions.
- **Public Key** – A key that anyone can use to verify the signature.

The private key should never be shared.

When Alice creates a transaction:

1. She signs it using her private key.
2. The signature is attached to the transaction.
3. Other nodes verify the signature using Alice's public key.

If the transaction is modified after signing, the signature becomes invalid.

This allows the network to verify ownership without ever knowing Alice's private key.

In [ ]:
!pip install cryptography

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
import json

# ----------------------------------------
# Generate Alice's Wallet
# ----------------------------------------

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print("Alice's wallet created!")

Alice's wallet created!


## Create a Transaction

In [ ]:
transaction = {
    "sender": "Alice",
    "receiver": "Bob",
    "amount": 20
}

transaction_json = json.dumps(
    transaction,
    sort_keys=True
).encode()

print(transaction)

{'sender': 'Alice', 'receiver': 'Bob', 'amount': 20}


Alice Signs It

In [ ]:
signature = private_key.sign(
    transaction_json,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Transaction signed!")

Transaction signed!


Verify the transacation Signed

In [ ]:
try:

    public_key.verify(
        signature,
        transaction_json,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("✅ Signature is VALID")

except Exception:

    print("❌ Signature is INVALID")

✅ Signature is VALID


# Attack Time
Let's modify the transaction

In [ ]:
transaction["amount"] = 1000

transaction_json = json.dumps(
    transaction,
    sort_keys=True
).encode()

In [ ]:
try:

    public_key.verify(
        signature,
        transaction_json,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("Valid")

except Exception:

    print("❌ Signature Invalid!")

❌ Signature Invalid!


Why id the Signature fail over here ?

No one can sign a fake unless they steal Alice's private Key

In [ ]:
bob_private = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

bob_public = bob_private.public_key()

In [ ]:
try:

    bob_public.verify(
        signature,
        transaction_json,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("Valid")

except Exception:

    print("❌ Bob cannot verify Alice's signature.")

❌ Bob cannot verify Alice's signature.


Now we have blocks. We have transactions. We have digital signatures. But where do new transactions go before they're added to a block?

# Section 7 – The Blockchain Class & Mempool

So far, we've manually created blocks and transactions.

A real blockchain needs something that manages:

- The blockchain (list of blocks)
- Pending transactions
- Creating the Genesis Block
- Adding new blocks
- Validating the chain

This is the job of the **Blockchain** class.

## What is a Mempool?

When a user creates a transaction, it is **not immediately added to the blockchain**.

Instead, it waits in a temporary area called the **Mempool** (Memory Pool).

```
Alice → Bob : 20
Bob → Charlie : 5
Alice → David : 15
```

↓

```
        MEMPOOL
+-------------------------+
| Alice → Bob : 20        |
| Bob → Charlie : 5       |
| Alice → David : 15      |
+-------------------------+
```

Miners select transactions from the mempool and include them in the next block.

This allows many users to submit transactions simultaneously before they are confirmed.

In [ ]:
import hashlib
import json
import time


class Block:

    def __init__(self, index, transactions, previous_hash):

        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.timestamp = time.time()

        # Used for Proof of Work
        self.nonce = 0

        # Initial hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):

        transaction_data = []

        for tx in self.transactions:

            if hasattr(tx, "to_dict"):
                transaction_data.append(tx.to_dict())
            else:
                transaction_data.append(tx)

        block_data = {
            "index": self.index,
            "transactions": transaction_data,
            "previous_hash": self.previous_hash,
            "timestamp": self.timestamp,
            "nonce": self.nonce
        }

        block_string = json.dumps(
            block_data,
            sort_keys=True
        ).encode()

        return hashlib.sha256(block_string).hexdigest()

    def mine_block(self, difficulty):

        target = "0" * difficulty

        print(f"\n⛏️ Mining Block #{self.index}")

        while not self.hash.startswith(target):

            self.nonce += 1
            self.hash = self.calculate_hash()

        print("\n✅ Block Successfully Mined!")
        print(f"Nonce : {self.nonce}")
        print(f"Hash  : {self.hash}")


    def __str__(self):

        return f"""
==========================
Block #{self.index}
==========================

Transactions:
{self.transactions}

Previous Hash:
{self.previous_hash}

Current Hash:
{self.hash}

Nonce:
{self.nonce}

Timestamp:
{self.timestamp}
"""


class Blockchain:

    def __init__(self):

        # List of blocks
        self.chain = []

        # Network mining difficulty
        self.difficulty = 4

        # Pending transactions waiting to be mined
        self.pending_transactions = []

        # Create the first block
        self.create_genesis_block()

    # ---------------------------------

    def create_genesis_block(self):

        genesis = Block(
            index=0,
            transactions=[],
            previous_hash="0"
        )

        self.chain.append(genesis)

    # ---------------------------------

    def add_transaction(self, transaction):

        self.pending_transactions.append(transaction)

        print(f"Transaction added to mempool.")
        print(f"Pending Transactions: {len(self.pending_transactions)}")

    # ---------------------------------

    def create_block(self):

        previous_block = self.chain[-1]

        block = Block(
            index=len(self.chain),
            transactions=self.pending_transactions.copy(),
            previous_hash=previous_block.hash
        )

        block.mine_block(self.difficulty)

        self.chain.append(block)

        # Clear mempool
        self.pending_transactions.clear()

        print(f"Block #{block.index} added!")

    def verify_chain(self):
      """
      Verify the integrity of the blockchain.
      """

      for i in range(1, len(self.chain)):

          current = self.chain[i]
          previous = self.chain[i - 1]

          # Verify the current block's hash
          if current.hash != current.calculate_hash():
              print(f"❌ Block {current.index} has been modified!")
              return False

          # Verify the link to the previous block
          if current.previous_hash != previous.hash:
              print(f"❌ Block {current.index} is not linked correctly!")
              return False

      print("✅ Blockchain is valid.")
      return True

    # ---------------------------------

    def print_chain(self):

        for block in self.chain:
            print(block)

    # ---------------------------------

    def print_mempool(self):

        print("\n========== MEMPOOL ==========")

        if len(self.pending_transactions) == 0:
            print("No pending transactions.")

        for tx in self.pending_transactions:
            print(tx)

        print("=============================\n")

Add Transactions

In [ ]:
tx1 = Transaction("Alice", "Bob", 20)
tx2 = Transaction("Bob", "Charlie", 10)
tx3 = Transaction("Charlie", "David", 5)

blockchain = Blockchain()

blockchain.add_transaction(tx1)
blockchain.add_transaction(tx2)
blockchain.add_transaction(tx3)

Transaction added to mempool.
Pending Transactions: 1
Transaction added to mempool.
Pending Transactions: 2
Transaction added to mempool.
Pending Transactions: 3


Show the Mempool

In [ ]:
blockchain.print_mempool()


========== MEMPOOL ==========

Transaction ID : 54b48af5-2761-4285-8d8b-6fbd62ac7801
Sender         : Alice
Receiver       : Bob
Amount         : $20
Timestamp      : 1785787964.670528


Transaction ID : aaa7e866-1298-47d9-90de-8b7a37b64096
Sender         : Bob
Receiver       : Charlie
Amount         : $10
Timestamp      : 1785787964.6706102


Transaction ID : ed708774-6117-42d0-80a6-31bfad3d474f
Sender         : Charlie
Receiver       : David
Amount         : $5
Timestamp      : 1785787964.6706862




Create a Block

In [ ]:
blockchain.create_block()
blockchain.print_chain()

# Printing the Mempool Again
blockchain.print_mempool()


⛏️ Mining Block #1

✅ Block Successfully Mined!
Nonce : 29169
Hash  : 000062d561287064cdb9ddf5e80394f09770ffa518ca734561b9e6f4937d4052
Block #1 added!

Block #0

Transactions:
[]

Previous Hash:
0

Current Hash:
0f2a1632bd1559913c15093ea39418bab8837d885c7205542b171ff717fa1363

Nonce:
0

Timestamp:
1785787964.6707456


Block #1

Transactions:
[<__main__.Transaction object at 0x7e85e3ec84d0>, <__main__.Transaction object at 0x7e85e3ecaab0>, <__main__.Transaction object at 0x7e85e3ecad80>]

Previous Hash:
0f2a1632bd1559913c15093ea39418bab8837d885c7205542b171ff717fa1363

Current Hash:
000062d561287064cdb9ddf5e80394f09770ffa518ca734561b9e6f4937d4052

Nonce:
29169

Timestamp:
1785787964.6832767


========== MEMPOOL ==========
No pending transactions.



Right now, how long does it take to create a block?
We can instanly create a block by calling blockchain.create_block()

# ⛏️ Section 8 – Mining (Proof of Work)

So far, creating a block is very easy.

An attacker could modify a block, recalculate its hash, and quickly rebuild the entire blockchain.

Bitcoin solves this using **Proof of Work (PoW).**

Before a block can be added to the blockchain, miners must solve a computational puzzle.

The puzzle is simple:

> Find a number (**nonce**) such that the block's hash starts with a certain number of leading zeros.

Example:

```
❌ 8f23ab1...
❌ 4ac812f...
❌ c2af891...
✅ 0000d91...
```

There is no shortcut.

The only solution is to keep trying different values until a valid hash is found.

This process is called **Mining**.

A nonce simply means:

Number used once

It is just another value inside the block.

Every time we change the nonce,

the hash changes.

## Demonstration of Nonce

In [ ]:
import hashlib
import json
import time


class Block:

    def __init__(self, index, transactions, previous_hash):

        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.timestamp = time.time()

        # Used for Proof of Work
        self.nonce = 0

        # Initial hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):

        transaction_data = []

        for tx in self.transactions:

            if hasattr(tx, "to_dict"):
                transaction_data.append(tx.to_dict())
            else:
                transaction_data.append(tx)

        block_data = {
            "index": self.index,
            "transactions": transaction_data,
            "previous_hash": self.previous_hash,
            "timestamp": self.timestamp,
            "nonce": self.nonce
        }

        block_string = json.dumps(
            block_data,
            sort_keys=True
        ).encode()

        return hashlib.sha256(block_string).hexdigest()

    def mine_block(self, difficulty):

        target = "0" * difficulty

        print(f"\n⛏️ Mining Block #{self.index}")

        while not self.hash.startswith(target):

            self.nonce += 1
            self.hash = self.calculate_hash()

        print("\n✅ Block Successfully Mined!")
        print(f"Nonce : {self.nonce}")
        print(f"Hash  : {self.hash}")

    def __str__(self):

        return f"""
==========================
Block #{self.index}
==========================

Transactions:
{self.transactions}

Previous Hash:
{self.previous_hash}

Current Hash:
{self.hash}

Nonce:
{self.nonce}

Timestamp:
{self.timestamp}
"""

Let's understand Nonce

In [ ]:
# Let's create a simple block

block = Block(
    index=1,
    transactions=["Alice -> Bob : $20"],
    previous_hash="0"
)

print("Current Nonce :", block.nonce)
print("Current Hash  :", block.hash)

Current Nonce : 0
Current Hash  : 45bf2d0e8b080ece3f912531d854f6f79825fe03cfb39d80c1b83e93576cf807


Change Only the Nonce

In [ ]:
print("Hash with Nonce = 0")
print(block.hash)

print()

block.nonce = 1

new_hash = block.calculate_hash()

print("Hash with Nonce = 1")
print(new_hash)

Hash with Nonce = 0
45bf2d0e8b080ece3f912531d854f6f79825fe03cfb39d80c1b83e93576cf807

Hash with Nonce = 1
5148c49b7523f8a5636f5bb33c26a6e622394945641364c2367e93f7e1bed90b


Difficulty Target

In [ ]:
difficulty = 4

target = "0" * difficulty

print("Difficulty :", difficulty)
print("We're looking for hashes that begin with:", target)
print("Target     :", target)

Difficulty : 4
We're looking for hashes that begin with: 0000
Target     : 0000


Same block. Only nonce changed. Entire hash changed.

Manual Mining

In [ ]:
block = Block(
    index=2,
    transactions=["Bob -> Charlie : $10"],
    previous_hash="0"
)

difficulty = 4
target = "0" * difficulty

attempts = 0

while not block.hash.startswith(target):

    block.nonce += 1
    block.hash = block.calculate_hash()

    attempts += 1

print("✅ Mining Complete!")
print(f"Attempts : {attempts}")
print(f"Nonce    : {block.nonce}")
print(f"Hash     : {block.hash}")

✅ Mining Complete!
Attempts : 95712
Nonce    : 95712
Hash     : 0000388faed39958395f0c01bf6e109df4a95d44b25bb403d45bc48cef460ec5


Let's Measure the Mining Time

In [ ]:
import time

block = Block(
    index=3,
    transactions=["Charlie -> David : $5"],
    previous_hash="0"
)

difficulty = 4

start = time.time()

while not block.hash.startswith("0" * difficulty):
    block.nonce += 1
    block.hash = block.calculate_hash()

end = time.time()

print("Mining Finished")
print(f"Nonce        : {block.nonce}")
print(f"Hash         : {block.hash}")
print(f"Time Taken   : {end-start:.2f} seconds")

Mining Finished
Nonce        : 748
Hash         : 0000872070b076c850cabeb2266d0974e06bccc101a447a2447d060bc60f9d32
Time Taken   : 0.01 seconds


What if we increase the difficulty to 6 ?
Did it take longer to mine ?
Yes, it needs to find 0000..... which is much harder

Final Mining Function.
We don't want to call this loop every time. Let's create a Block for mining.

In [ ]:
def mine_block(self, difficulty):
    """
    Perform Proof of Work by finding a hash
    with the required number of leading zeros.
    """

    target = "0" * difficulty

    while not self.hash.startswith(target):
        self.nonce += 1
        self.hash = self.calculate_hash()

    print(f"✅ Block #{self.index} mined!")
    print(f"Nonce : {self.nonce}")
    print(f"Hash  : {self.hash}")

In [ ]:
block = Block(
    index=4,
    transactions=["David -> Eve : $25"],
    previous_hash="0"
)

block.mine_block(difficulty=4)


⛏️ Mining Block #4

✅ Block Successfully Mined!
Nonce : 1
Hash  : 00007273d2ff2c57d969ddca8bd93953cf1f8fda61d1792f269cdcba0f3a5131


Integrating with the Blockchain

### Section 8 Summary

- Proof of Work (PoW) makes creating a block computationally expensive.
- Miners repeatedly change the **nonce** until the block's hash satisfies the required difficulty (leading zeros).
- There is no shortcut the only approach is trial and error (brute force).
- Increasing the mining difficulty increases the average time required to find a valid hash.
- If an attacker modifies a mined block, they must re-mine that block and every subsequent block.
- Mining, together with linked blocks, helps make blockchain history difficult to rewrite.

**Key Takeaway:** Proof of Work doesn't prevent changes to the blockchain it makes changing history computationally expensive.

# 🌐 Section 9 – Blockchain Nodes

So far, we've built a blockchain that runs on a single computer.

But a blockchain isn't useful if only one computer stores it.

Instead, many computers (called **nodes**) each maintain their own copy of the blockchain.

Every node is responsible for:

- Storing a copy of the blockchain.
- Verifying transactions.
- Verifying newly mined blocks.
- Sharing valid blocks with other nodes.

There is no central server.

Each node independently verifies information before accepting it.

## Network Overview

```
        Node A
           │
     ┌─────┼─────┐
     │     │     │
   Node B Node C Node D
```

When one node mines a valid block, it broadcasts that block to the rest of the network.

Each receiving node verifies the block before adding it to its own blockchain.

Simulate Multiple Nodes

In [ ]:
# Simulate three blockchain nodes

node_a = Blockchain()
node_b = Blockchain()
node_c = Blockchain()

print("Three blockchain nodes created.")

Three blockchain nodes created.


Each node/independent computer has it's own blockchain and mempool.
Everything starts with the Genesis Block.

Mine on One Node

In [ ]:
tx = Transaction("Alice", "Bob", 25)

node_a.add_transaction(tx)
#let's change the difficulty level

blockchain.difficulty = 5
node_a.create_block()

Transaction added to mempool.
Pending Transactions: 1

⛏️ Mining Block #1

✅ Block Successfully Mined!
Nonce : 26480
Hash  : 00006a340476adeb1e5f2a9b70ce2269a3c5f83d039438d7e721949e372e615d
Block #1 added!


Let's compare the chains

In [ ]:
print("Node A Chain Length:", len(node_a.chain))
print("Node B Chain Length:", len(node_b.chain))
print("Node C Chain Length:", len(node_c.chain))

Node A Chain Length: 2
Node B Chain Length: 1
Node C Chain Length: 1


As, we can see that Node A hasn't shared the block yet.

Let's simulate Broadcasting which allows nodes to send blocks over internet

In [ ]:
import copy

node_b.chain = copy.deepcopy(node_a.chain)
node_c.chain = copy.deepcopy(node_a.chain)

print("Block broadcasted to the network.")

Block broadcasted to the network.


In [ ]:
print("Node A:", len(node_a.chain))
print("Node B:", len(node_b.chain))
print("Node C:", len(node_c.chain))

Node A: 2
Node B: 2
Node C: 2


You might be wondering

why did we use deepcopy()?

because both nodes would point to the same Python list.
Real computers don't share memory.
Each node has it's own independent copy of the blockchain.
deepcop() helps in better represents how a network behaves.

**Nodes blindly don't trust each other.**

They trust verification.

Every node independently checks:

Is the block hash valid?
Does the previous hash match?
Is the Proof of Work valid?

Only then is the block accepted.

### What happens if Node A and Node B both mine a different valid block at almost the same time?

        Genesis
            │
        Block 1
         /    \
    Block 2A  Block 2B

  Now there are two valid blockchains.

  Which one should everyone follow ?

  That's where **Consensus** solves

### Section 9 Summary

- A blockchain network consists of many independent nodes.
- Every node stores its own copy of the blockchain.
- Nodes independently verify transactions and blocks.
- When a node mines a new block, it broadcasts it to other nodes.
- Other nodes verify the block before accepting it.
- Nodes trust cryptographic verification, not each other.

**Key Takeaway:** A blockchain is not one computer it is a network of independent computers that maintain and verify the same ledger.

# 🤝 Section 10 – Consensus

Imagine two miners solve the Proof of Work puzzle at nearly the same time.

Both create a valid block.

Now the network temporarily has **two different versions of the blockchain**.

```
            Genesis
               │
            Block 1
            /      \
      Block 2A    Block 2B
```

This is called a **fork**.

Different nodes may receive different blocks first, so they temporarily disagree on the latest block.

Bitcoin resolves this using a consensus rule:

> **Follow the valid chain with the greatest cumulative Proof of Work.**

As new blocks are mined, one branch becomes longer (has more accumulated work), and the network gradually converges on a single chain.

Consensus is not about trusting miners it is about every node independently applying the same validation rules and choosing the chain with the most work.

## If Node A mined Block 2A and Node B mined Block 2B at exactly the same time... who's right?

Both blocks satisfy the Proof of Work.

Both are valid.

The network simply hasn't agreed yet.

Fork Blockchain

In [ ]:
import copy

# Start with the same blockchain
node_a = Blockchain()
node_b = copy.deepcopy(node_a)

print("Both nodes start with the same blockchain.")
print(f"Node A Chain Length: {len(node_a.chain)}")
print(f"Node B Chain Length: {len(node_b.chain)}")

Both nodes start with the same blockchain.
Node A Chain Length: 1
Node B Chain Length: 1


Mine Different Blocks

In [ ]:
# Node A mines a block

node_a.add_transaction(
    Transaction("Alice", "Bob", 25)
)
blockchain.difficulty = 7
node_a.create_block()


# Node B mines a different block

node_b.add_transaction(
    Transaction("Charlie", "David", 10)
)
blockchain.difficulty = 7
node_b.create_block()

Transaction added to mempool.
Pending Transactions: 1

⛏️ Mining Block #1

✅ Block Successfully Mined!
Nonce : 77552
Hash  : 0000619429a43a550298480ad4923b9874aa988b107a3d096baa7e816b8a7bf8
Block #1 added!
Transaction added to mempool.
Pending Transactions: 1

⛏️ Mining Block #1

✅ Block Successfully Mined!
Nonce : 12780
Hash  : 000053cb078c30039ee4fd90587fb9f4e4665eb2a5dd3dbc007acb788a9c2a4c
Block #1 added!


Both miners worked independently.

Both produced valid blocks.

Compare Chains

In [ ]:
print("Node A Latest Transaction:")
print(node_a.chain[-1].transactions[0])

print()

print("Node B Latest Transaction:")
print(node_b.chain[-1].transactions[0])

Node A Latest Transaction:

Transaction ID : dcecc5f3-f823-4590-a33d-4970f8b680e2
Sender         : Alice
Receiver       : Bob
Amount         : $25
Timestamp      : 1785787967.169933


Node B Latest Transaction:

Transaction ID : b09b1c4f-6873-4647-9caf-01e1a58dafe7
Sender         : Charlie
Receiver       : David
Amount         : $10
Timestamp      : 1785787968.474585



Node A

Genesis

↓

Block 1

↓

Alice → Bob


------------------


Node B

Genesis

↓

Block 1

↓

Charlie → David

Both are valid blocks.

One Node Mines Again

Suppose Node A mines another block first

In [ ]:
node_a.add_transaction(
    Transaction("Bob", "Charlie", 15)
)

node_a.difficulty = 4

node_a.create_block()

print(len(node_a.chain))
print(len(node_b.chain))

Transaction added to mempool.
Pending Transactions: 1

⛏️ Mining Block #2

✅ Block Successfully Mined!
Nonce : 177998
Hash  : 0000fa5835e11e3b47a61b6757b273063f1ee76be01dbba50c2c6b047f123092
Block #2 added!
3
2


Which block would you trust ?
Node A
Why ?
Because it contains more cumulative Proof of Work.

Simulate Consensus

In [ ]:
if len(node_a.chain) > len(node_b.chain):

    node_b.chain = copy.deepcopy(node_a.chain)

    print("Node B switched to Node A's chain.")


print(len(node_a.chain))
print(len(node_b.chain))

Node B switched to Node A's chain.
3
3


Without consensus

Everyone would have different balances.

Consensus ensures the network eventually agrees on one shared history.


### Section 10 Summary

- Multiple miners can produce valid blocks at nearly the same time.
- This creates a temporary fork in the blockchain.
- Different nodes may temporarily follow different branches.
- As additional blocks are mined, one branch accumulates more Proof of Work.
- Nodes switch to the valid chain with the greatest cumulative Proof of Work.
- The network eventually converges on a single shared blockchain.

**Key Takeaway:** Consensus is the process by which independent nodes eventually agree on one blockchain, even when temporary forks occur.

# Section 11 – Build & Break: Common Blockchain Attacks

We've built a simple Layer 1 blockchain with:

- Blocks
- Hashing
- Linked blocks
- Transactions
- Digital signatures
- Mempool
- Mining (Proof of Work)
- Nodes
- Consensus

Now let's think like an attacker.

We'll look at some common attacks against blockchains and understand how the mechanisms we've implemented help defend against them.

> **Important:** No system is perfectly secure. Blockchain security comes from combining multiple mechanisms not from any single feature.

## Transaction Tampering

In [ ]:
print("Original Transaction:")
print(blockchain.chain[1].transactions[0])

# Attacker modifies the transaction
blockchain.chain[1].transactions[0].amount = 999999

print("\nModified Transaction:")
print(blockchain.chain[1].transactions[0])

print("\nVerifying Blockchain...")
blockchain.verify_chain()

Original Transaction:

Transaction ID : 54b48af5-2761-4285-8d8b-6fbd62ac7801
Sender         : Alice
Receiver       : Bob
Amount         : $20
Timestamp      : 1785787964.670528


Modified Transaction:

Transaction ID : 54b48af5-2761-4285-8d8b-6fbd62ac7801
Sender         : Alice
Receiver       : Bob
Amount         : $999999
Timestamp      : 1785787964.670528


Verifying Blockchain...
❌ Block 1 has been modified!


False

Why did verification fail?

## Fake Transactions

What's stopping me from creating this?

## Double Spending

Alice owns 10 coins.

She creates:

Alice → Bob : 10

At the same time:

Alice → Charlie : 10

If both transactions were accepted, Alice would spend the same coins twice.

Consensus ensures that only one transaction becomes part of the accepted blockchain.

## 51% Attack



If an attacker controls more than 50% of the network's mining power, they may be able to build an alternative chain faster than the honest network.

## Replay Attack

Suppose Alice signs:
> Alice → Bob : 10

If the transaction lacks unique identifiers or replay protection, an attacker might try to submit the exact same signed transaction again.

Explain that real blockchains prevent this using mechanisms such as:

*   Transaction IDs
*   Nonces
*   Account sequence numbers (depending on the blockchain design)

### Workshop Summary

Today we built a simplified Layer 1 blockchain from scratch.

We implemented:

- A Block class
- SHA-256 hashing
- Linked blocks
- Transactions
- Digital signatures
- Blockchain class
- Mempool
- Proof of Work
- Multiple nodes
- Basic consensus

We also explored how these components work together to defend against common attacks.

### Key Takeaways

- A blockchain is a distributed ledger.
- Hashes make tampering detectable.
- Previous hashes link blocks into a chain.
- Digital signatures prove transaction ownership.
- Mining makes rewriting history computationally expensive.
- Nodes independently verify blocks.
- Consensus allows the network to agree on a single shared history.
- Security comes from combining all these mechanisms—not from any single feature.